In [1]:
import pandas as pd
import yfinance as yf
from datetime import date
from pathlib import Path

fin = date.today().isoformat()
PAIRES = {"HKD": "HKDUSD=X", "EUR": "EURUSD=X"}

taux = []
for devise, ticker in PAIRES.items():
    fx = yf.download(ticker, start="2015-01-01", end=fin,
                     interval="1d", auto_adjust=True, progress=False)
    if isinstance(fx.columns, pd.MultiIndex):
        fx.columns = fx.columns.get_level_values(0)
    fx = fx.reset_index()[["Date", "Close"]]
    fx.columns = ["date", "taux_usd"]
    fx["devise"] = devise
    taux.append(fx)

# L'USD vaut 1 par construction : sans cette ligne, les valeurs
# américaines disparaîtraient à la jointure.
usd = pd.DataFrame({"date": taux[0]["date"], "taux_usd": 1.0, "devise": "USD"})
taux.append(usd)

fx_all = pd.concat(taux, ignore_index=True)

Path("data/bronze_fx").mkdir(parents=True, exist_ok=True)
fx_all.to_parquet("data/bronze_fx/taux.parquet", index=False)
print(fx_all.groupby("devise").size())

devise
EUR    3047
HKD    3047
USD    3047
dtype: int64
